In [108]:
import os
from pathlib import Path
import uuid
import re

import pandas as pd
import numpy as np
from unidecode import unidecode


**CAREFULL**  
This is the correct notebook that generates the dataset_no0min.csv 

In [109]:
# Paths
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

TRAINING_PATH = OUTPUT_DIR / "training_data_v2.csv"
UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping.csv"
CLEANED_UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping_cleaned.csv"
MASTER_TEAM_LIST_PATH = Path("master_team_list.csv")

print(" DATA_ROOT:", DATA_ROOT.resolve())
print(" OUTPUT_DIR:", OUTPUT_DIR.resolve())
print(" training_data_v2 path:", TRAINING_PATH)


 DATA_ROOT: C:\Python\fpl_pipeline\data
 OUTPUT_DIR: C:\Python\fpl_pipeline\output
 training_data_v2 path: output\training_data_v2.csv


In [110]:
# %%– Normalization function
def normalize_player_name(name: str) -> str:
    """
    Normalizes player names:
    - to lowercase
    - remove accents
    - strip trailing numbers (e.g. ' 534')
    - remove underscores
    - keep only alphanumeric + spaces
    - collapse multiple spaces
    """
    if pd.isna(name):
        return name

    name = str(name).strip().lower()
    name = unidecode(name)

    # Remove trailing numeric suffixes: "aaron connolly 534", "aaron_connolly_534"
    name = re.sub(r'[\s_]*\d+\s*$', '', name)

    # Replace underscores with spaces
    name = name.replace("_", " ")

    # Keep only alphanumeric + space
    name = "".join(c for c in name if c.isalnum() or c.isspace())

    # Collapse multiple spaces
    name = " ".join(name.split())

    return name


print(" Testing normalize_player_name:")
for t in ["aaron cresswell 376", "Aaron_Connolly534", "Son Heung-Min 123"]:
    print(f"  '{t}' -> '{normalize_player_name(t)}'")


 Testing normalize_player_name:
  'aaron cresswell 376' -> 'aaron cresswell'
  'Aaron_Connolly534' -> 'aaron connolly'
  'Son Heung-Min 123' -> 'son heungmin'


In [111]:
# Updated Loading function for 2020-2026
def load_all_gws(data_root=DATA_ROOT):
    all_seasons = []
    # Explicit list of seasons to include
    target_seasons = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
    
    print(f"Loading GW data for seasons: {target_seasons}")

    for season in target_seasons:
        season_path = data_root / season
        gws_path = season_path / "gws"

        if not season_path.is_dir() or not gws_path.exists():
            print(f"Skipping {season}: Folder or 'gws' subfolder not found.")
            continue

        gw_files = sorted(
            [f for f in os.listdir(gws_path) if f.startswith("gw") and f.endswith(".csv")],
            key=lambda x: int(x.replace("gw","").replace(".csv",""))
        )

        print(f"Season {season}: {len(gw_files)} GWs found.")

        frames = []
        for fname in gw_files:
            gw = int(fname.replace("gw","").replace(".csv",""))
            df = pd.read_csv(gws_path / fname)

            # Standardized columns for consistency across seasons
            keep = [
                "name", "element", "minutes", "goals_scored", "assists", "clean_sheets",
                "goals_conceded","yellow_cards","red_cards","total_points",
                "influence","creativity","threat","ict_index",
                "opponent_team","was_home"
            ]
            cols = [c for c in keep if c in df.columns]
            df = df[cols].copy()

            df["season"] = season
            df["Gameweek"] = gw
            frames.append(df)

        if frames:
            all_seasons.append(pd.concat(frames, ignore_index=True))

    if not all_seasons:
        raise ValueError("No data was loaded. Check your DATA_ROOT path.")
        
    df_gws = pd.concat(all_seasons, ignore_index=True)
    print(f"Total rows loaded: {len(df_gws):,}")
    return df_gws

df_gws = load_all_gws()
display(df_gws.head())
print(df_gws["season"].value_counts().sort_index())


Loading GW data for seasons: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Season 2020-21: 38 GWs found.
Season 2021-22: 38 GWs found.
Season 2022-23: 38 GWs found.


C:\Users\SOFI\AppData\Local\Temp\ipykernel_11324\3985452101.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_seasons.append(pd.concat(frames, ignore_index=True))


Season 2023-24: 38 GWs found.
Season 2024-25: 38 GWs found.
Season 2025-26: 32 GWs found.
Total rows loaded: 158,463


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,influence,creativity,threat,ict_index,opponent_team,was_home,season,Gameweek
0,Aaron Connolly,78,45,0,0,0,2,0,0,1,1.2,0.3,32.0,3.4,5,True,2020-21,1
1,Aaron Cresswell,435,90,0,0,0,2,0,0,1,10.4,11.2,0.0,2.2,14,True,2020-21,1
2,Aaron Mooy,60,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,5,True,2020-21,1
3,Aaron Ramsdale,483,90,0,0,0,2,0,0,1,18.2,0.0,0.0,1.8,20,True,2020-21,1
4,Abdoulaye Doucouré,512,90,0,0,1,0,0,0,3,20.4,44.6,4.0,6.9,17,False,2020-21,1


season
2020-21    24365
2021-22    25447
2022-23    26505
2023-24    29725
2024-25    27605
2025-26    24816
Name: count, dtype: int64


In [112]:
#– Φόρτωμα players_raw ανά σεζόν
def load_players_raw_by_season():
    players = {}
    print("\n Loading players_raw per season...")

    for season in sorted(df_gws["season"].unique()):
        path = DATA_ROOT / season / "players_raw.csv"
        if not path.exists():
            print(f"  ⚠️ No players_raw for {season}")
            continue

        df = pd.read_csv(path)

        # unify ID col
        if "id" in df.columns:
            df.rename(columns={"id": "element"}, inplace=True)

        keep = ["element","team","element_type","web_name","first_name","second_name"]
        keep = [c for c in keep if c in df.columns]
        df = df[keep].copy()

        df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
        if "team" in df.columns:
            df["team"] = pd.to_numeric(df["team"], errors="coerce").astype("Int64")

        players[season] = df.set_index("element")
        print(f"  → {season}: {len(df)} players in players_raw")

    return players


players_raw_by_season = load_players_raw_by_season()



 Loading players_raw per season...
  → 2020-21: 713 players in players_raw
  → 2021-22: 737 players in players_raw
  → 2022-23: 778 players in players_raw
  → 2023-24: 865 players in players_raw
  → 2024-25: 804 players in players_raw
  → 2025-26: 826 players in players_raw


In [113]:
# %% Teams per season (με master_team_list fallback για 2018-19)
master_teams_df = pd.read_csv(MASTER_TEAM_LIST_PATH) if MASTER_TEAM_LIST_PATH.exists() else None
if master_teams_df is not None:
    master_teams_df.columns = [c.lower() for c in master_teams_df.columns]

def load_teams_for_season(season: str) -> pd.DataFrame:
    """
    Επιστρέφει DF με στήλες: Team ID, Team Name, short_name
    - teams.csv αν υπάρχει
    - αλλιώς master_team_list.csv (π.χ. 2018-19)
    """
    path = DATA_ROOT / season / "teams.csv"

    if path.exists():
        df = pd.read_csv(path)
        idcol = "id" if "id" in df.columns else "code"
        df.rename(columns={idcol: "Team ID", "name": "Team Name"}, inplace=True)
        df["Team ID"] = pd.to_numeric(df["Team ID"], errors="coerce").astype("Int64")
        if "short_name" not in df.columns:
            df["short_name"] = df["Team Name"]
        return df[["Team ID", "Team Name", "short_name"]]

    if master_teams_df is not None:
        sub = master_teams_df[master_teams_df["season"] == season].copy()
        if not sub.empty:
            sub.rename(columns={"team": "Team ID", "team_name": "Team Name"}, inplace=True)
            sub["Team ID"] = pd.to_numeric(sub["Team ID"], errors="coerce").astype("Int64")
            sub["short_name"] = sub["Team Name"]
            print(f"  🔁 Using master_team_list for {season}")
            return sub[["Team ID", "Team Name", "short_name"]]

    raise RuntimeError(f"❌ No team info for {season}")


In [114]:
#– Φόρτωμα fixtures σε “long” μορφή (home/away rows)
def load_fixtures_long():
    rows = []
    print("\n Building fixture_long from fixtures.csv ...")

    for season in sorted(df_gws["season"].unique()):
        fx_path = DATA_ROOT / season / "fixtures.csv"
        if not fx_path.exists():
            print(f"  ⚠️ No fixtures.csv for {season}, skipping.")
            continue

        fx = pd.read_csv(fx_path)
        if "event" in fx.columns:
            fx["Gameweek"] = fx["event"]
        elif "round" in fx.columns:
            fx["Gameweek"] = fx["round"]
        else:
            raise RuntimeError(f"No event/round column in fixtures for {season}")

        for _, r in fx.iterrows():
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_h"),
                "Opponent ID": r.get("team_a"),
                "Is Home": True,
                "Difficulty": r.get("team_h_difficulty", np.nan)
            })
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_a"),
                "Opponent ID": r.get("team_h"),
                "Is Home": False,
                "Difficulty": r.get("team_a_difficulty", np.nan)
            })

    fixture_long = pd.DataFrame(rows)
    fixture_long["Gameweek"] = pd.to_numeric(fixture_long["Gameweek"], errors="coerce").astype("Int64")
    fixture_long["Team ID"] = pd.to_numeric(fixture_long["Team ID"], errors="coerce").astype("Int64")
    fixture_long["Opponent ID"] = pd.to_numeric(fixture_long["Opponent ID"], errors="coerce").astype("Int64")
    fixture_long["Is Home"] = fixture_long["Is Home"].astype(bool)

    print("fixture_long rows:", len(fixture_long))
    return fixture_long


fixture_long = load_fixtures_long()
display(fixture_long.head())



 Building fixture_long from fixtures.csv ...
fixture_long rows: 4560


,season,Gameweek,Team ID,Opponent ID,Is Home,Difficulty
0,2020-21,1,8,1,True,3
1,2020-21,1,1,8,False,2
2,2020-21,1,6,16,True,2
3,2020-21,1,16,6,False,3
4,2020-21,1,11,10,True,3


In [115]:
# Build df_core per season by deriving Player Team ID from fixtures (perfect fix)
print("Building df_core per season using fixture-based team identification...")

def build_season_core(season: str, df_season: pd.DataFrame) -> pd.DataFrame:
    print(f"Season {season}...")

    df = df_season.copy()
    df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
    df["Gameweek"] = pd.to_numeric(df["Gameweek"], errors="coerce").astype("Int64")
    df["opponent_team"] = pd.to_numeric(df["opponent_team"], errors="coerce").astype("Int64")

    # players_raw for names, element_type ONLY (NOT team)
    pr = players_raw_by_season.get(season)
    df = df.merge(
        pr[["element_type", "web_name", "first_name", "second_name"]],
        left_on="element", right_index=True, how="left"
    )

    # fixtures for true team mapping
    fx_path = DATA_ROOT / season / "fixtures.csv"
    fx = pd.read_csv(fx_path)
    
    # Identify Gameweek column
    gw_col = "event" if "event" in fx.columns else "round"
    
    # Convert and drop rows where Gameweek or Teams are missing
    fx[gw_col] = pd.to_numeric(fx[gw_col], errors="coerce")
    fx = fx.dropna(subset=[gw_col, "team_h", "team_a"])

    # reconstruct Player Team ID + Opponent Difficulty by matching opponent_team
    rows = []
    for _, r in fx.iterrows():
        # Using int() only after ensuring no NaNs exist in these specific rows
        gw = int(r[gw_col])
        h = int(r["team_h"])
        a = int(r["team_a"])
        dh = int(r["team_h_difficulty"]) if pd.notna(r["team_h_difficulty"]) else 0
        da = int(r["team_a_difficulty"]) if pd.notna(r["team_a_difficulty"]) else 0

        rows.append({"Gameweek": gw, "OppKey": a, "Player Team ID": h, "Opponent ID": a, "Is Home": True, "Opponent Difficulty": dh})
        rows.append({"Gameweek": gw, "OppKey": h, "Player Team ID": a, "Opponent ID": h, "Is Home": False, "Opponent Difficulty": da})

    opp_map = pd.DataFrame(rows)
    opp_map["Gameweek"] = opp_map["Gameweek"].astype("Int64")
    opp_map["OppKey"] = opp_map["OppKey"].astype("Int64")

    df = df.merge(
        opp_map,
        left_on=["Gameweek", "opponent_team"],
        right_on=["Gameweek", "OppKey"],
        how="left"
    ).drop(columns=["OppKey"])

    # team names
    teams_df = load_teams_for_season(season)

    df = df.merge(
        teams_df[["Team ID", "Team Name"]],
        left_on="Player Team ID",
        right_on="Team ID",
        how="left"
    ).rename(columns={"Team Name": "Player Team Name"}).drop(columns=["Team ID"])

    df = df.merge(
        teams_df.rename(columns={"Team ID": "Opponent ID", "Team Name": "Opponent Name"})[["Opponent ID", "Opponent Name"]],
        on="Opponent ID",
        how="left"
    )

    # clean names
    df["Player Name"] = (
        df["first_name"].fillna("") + " " + df["second_name"].fillna("")
    ).str.strip().replace("", np.nan).fillna(df["name"])
    df["Web Name"] = df["web_name"]

    return df

# Build the core frames for all selected seasons
core_frames = [build_season_core(s, df_gws[df_gws["season"] == s]) for s in sorted(df_gws["season"].unique())]
df_core = pd.concat(core_frames, ignore_index=True)

print("df_core rebuilt with 100% accurate Player Team IDs")
display(df_core.head())

Building df_core per season using fixture-based team identification...
Season 2020-21...
Season 2021-22...
Season 2022-23...
Season 2023-24...
Season 2024-25...
Season 2025-26...
df_core rebuilt with 100% accurate Player Team IDs


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,...,first_name,second_name,Player Team ID,Opponent ID,Is Home,Opponent Difficulty,Player Team Name,Opponent Name,Player Name,Web Name
0,Aaron Connolly,78,45,0,0,0,2,0,0,1,...,Aaron,Connolly,3.0,5.0,True,4.0,Brighton,Chelsea,Aaron Connolly,Connolly
1,Aaron Cresswell,435,90,0,0,0,2,0,0,1,...,Aaron,Cresswell,19.0,14.0,True,2.0,West Ham,Newcastle,Aaron Cresswell,Cresswell
2,Aaron Mooy,60,0,0,0,0,0,0,0,0,...,Aaron,Mooy,3.0,5.0,True,4.0,Brighton,Chelsea,Aaron Mooy,Mooy
3,Aaron Ramsdale,483,90,0,0,0,2,0,0,1,...,Aaron,Ramsdale,15.0,20.0,True,2.0,Sheffield Utd,Wolves,Aaron Ramsdale,Ramsdale
4,Abdoulaye Doucouré,512,90,0,0,1,0,0,0,3,...,Abdoulaye,Doucouré,7.0,17.0,False,4.0,Everton,Spurs,Abdoulaye Doucouré,Doucouré


In [116]:
# %%– Diagnostic: Opponent Difficulty completeness
print(" Opponent Difficulty — overall summary (df_core)")
total_rows = len(df_core)
missing_od = df_core["Opponent Difficulty"].isna().sum()
print(f"  Total rows: {total_rows:,}")
print(f"  Missing Opponent Difficulty: {missing_od:,} ({missing_od/total_rows*100:.4f}%)")

print("\n🔍 Missing Opponent Difficulty by season:")
season_stats = (
    df_core
    .groupby("season")["Opponent Difficulty"]
    .apply(lambda s: s.isna().sum())
    .to_frame("Missing_OD")
)
season_stats["Total_Rows"] = df_core.groupby("season")["Opponent Difficulty"].size()
season_stats["Missing_%"] = (season_stats["Missing_OD"] / season_stats["Total_Rows"] * 100).round(4)

display(season_stats)


 Opponent Difficulty — overall summary (df_core)
  Total rows: 171,905
  Missing Opponent Difficulty: 161 (0.0937%)

🔍 Missing Opponent Difficulty by season:


,Missing_OD,Total_Rows,Missing_%
season,,,
2020-21,0,27397,0.0000
2021-22,0,29837,0.0000
2022-23,0,29618,0.0000
2023-24,0,31707,0.0000
2024-25,0,28372,0.0000
2025-26,161,24974,0.6447


In [117]:
# Cleaning & canonical columns
POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

rename_map = {
    "element": "Code",
    "minutes": "Minutes Played",
    "goals_scored": "Goals Scored",
    "assists": "Assists",
    "clean_sheets": "Clean Sheet",
    "goals_conceded": "Goals Conceded",
    "yellow_cards": "Yellow Card",
    "red_cards": "Red Cards",
    "total_points": "Total Points",
    "influence": "Influence",
    "creativity": "Creativity",
    "threat": "Threat",
    "ict_index": "ICT Index",
}

df_clean = df_core.copy()
df_clean.rename(columns=rename_map, inplace=True)

# Position από element_type
if "element_type" in df_clean.columns:
    df_clean["Position"] = df_clean["element_type"].map(POS_MAP)
else:
    df_clean["Position"] = np.nan

# Normalized name
df_clean["Player Name Norm"] = df_clean["Player Name"].apply(normalize_player_name)

print("Sample cleaned rows:")
display(df_clean[[
    "Player Name", "Player Name Norm", "Player Team Name",
    "Opponent Name", "Total Points"
]].head())


Sample cleaned rows:


,Player Name,Player Name Norm,Player Team Name,Opponent Name,Total Points
0,Aaron Connolly,aaron connolly,Brighton,Chelsea,1
1,Aaron Cresswell,aaron cresswell,West Ham,Newcastle,1
2,Aaron Mooy,aaron mooy,Brighton,Chelsea,0
3,Aaron Ramsdale,aaron ramsdale,Sheffield Utd,Wolves,1
4,Abdoulaye Doucouré,abdoulaye doucoure,Everton,Spurs,3


In [118]:
# Injury flag (3+ συνεχόμενα 0 λεπτά)
df_clean = df_clean.sort_values(["Player Name Norm","season","Gameweek"])
df_clean["Injury/Unavailable"] = 0

for p in df_clean["Player Name Norm"].unique():
    mask = df_clean["Player Name Norm"] == p
    m = df_clean.loc[mask, "Minutes Played"]

    streak = 0
    flags = []
    for x in m:
        if x == 0:
            streak += 1
            flags.append(1 if streak >= 3 else 0)
        else:
            streak = 0
            flags.append(0)
    df_clean.loc[mask, "Injury/Unavailable"] = flags

df_clean[["Player Name", "season", "Gameweek", "Minutes Played", "Injury/Unavailable"]].head(15)


,Player Name,season,Gameweek,Minutes Played,Injury/Unavailable
136097,Aaron Anselmino,2024-25,25,0,0
136908,Aaron Anselmino,2024-25,26,0,0
137691,Aaron Anselmino,2024-25,27,0,1
138479,Aaron Anselmino,2024-25,28,0,1
139137,Aaron Anselmino,2024-25,29,0,1
139909,Aaron Anselmino,2024-25,30,0,1
140701,Aaron Anselmino,2024-25,31,0,1
141694,Aaron Anselmino,2024-25,32,0,1
142918,Aaron Anselmino,2024-25,33,0,1
143644,Aaron Anselmino,2024-25,34,0,1


In [119]:
#Rolling averages L3 & L5
def add_lagged(df):
    metrics = [
        "Total Points", "Minutes Played", "Goals Scored", "Assists",
        "Goals Conceded", "ICT Index", "Threat", "Creativity", "Influence"
    ]

    df = df.sort_values(["Player Name Norm", "season", "Gameweek"])

    for w in [3, 5]:
        for c in metrics:
            if c not in df.columns:
                continue
            new = f"Avg_{c}_L{w}"
            df[new] = (
                df.groupby(["Player Name Norm","season"])[c]
                .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
                .fillna(0)
            )
    return df


df_lagged = add_lagged(df_clean)
display(df_lagged.head())


,name,Code,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Yellow Card,Red Cards,Total Points,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
136097,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136908,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137691,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138479,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
139137,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [120]:
# UUID mapping (χρησιμοποιεί cleaned αν υπάρχει)
if CLEANED_UUID_MAPPING_PATH.exists():
    print(" Using cleaned UUID mapping:", CLEANED_UUID_MAPPING_PATH)
    mapping = pd.read_csv(CLEANED_UUID_MAPPING_PATH)

    norm_col = [c for c in mapping.columns if c.lower() == "player name norm"][0]
    uuid_col = [c for c in mapping.columns if c.lower() == "player uuid"][0]

    uuid_map = dict(zip(mapping[norm_col], mapping[uuid_col]))

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(uuid_map)

    print("UUIDs filled:", df_lagged["Player UUID"].notna().sum())

else:
    print("No cleaned mapping. Creating NEW stable mapping…")

    unique_norms = sorted(df_lagged["Player Name Norm"].unique())
    new_uuids = [str(uuid.uuid4()) for _ in unique_norms]

    mapping = pd.DataFrame({
        "Player Name Norm": unique_norms,
        "Player UUID": new_uuids
    })

    rep = df_lagged.groupby("Player Name Norm")[["Player Name", "Web Name"]] \
        .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan).reset_index()

    mapping = mapping.merge(rep, on="Player Name Norm", how="left")

    mapping.to_csv(UUID_MAPPING_PATH, index=False, encoding="utf-8-sig")
    print(f" Saved new UUID mapping → {UUID_MAPPING_PATH}")

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(
        dict(zip(mapping["Player Name Norm"], mapping["Player UUID"]))
    )

display(df_lagged[["Player Name","Player Name Norm","Player UUID"]].head())


 Using cleaned UUID mapping: output\player_uuid_mapping_cleaned.csv
UUIDs filled: 163913


,Player Name,Player Name Norm,Player UUID
136097,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
136908,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
137691,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
138479,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
139137,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815


In [121]:
#Τελικό save σε training_data_v2.csv
base_cols = [
    "Player UUID", "Code", "Player Name", "Web Name", "Player Team Name",
    "season", "Gameweek", "Minutes Played", "Goals Scored", "Assists",
    "Clean Sheet", "Goals Conceded", "Yellow Card", "Red Cards",
    "Total Points", "Threat", "ICT Index", "Influence", "Creativity",
    "Opponent Name", "Opponent Difficulty", "Is Home", "Position",
    "Injury/Unavailable"
]

lagged_cols = [c for c in df_lagged.columns if c.startswith("Avg_")]
final_cols = base_cols + lagged_cols

df_final = df_lagged[final_cols].copy()

df_final.to_csv(TRAINING_PATH, index=False, encoding="utf-8-sig")

print(" training_data_v2.csv CREATED!")
print("Rows:", len(df_final))
print("Cols:", len(df_final.columns))
display(df_final.head())


 training_data_v2.csv CREATED!
Rows: 171905
Cols: 42


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
136097,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,25,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136908,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,26,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137691,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,27,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138479,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,28,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
139137,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,29,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [122]:
#Sanity checks
print("Unique seasons:", sorted(df_final["season"].unique()))
print("Max GW per season:", df_final.groupby("season")["Gameweek"].max().to_dict())

print("\nOpponent Difficulty non-null:", df_final["Opponent Difficulty"].notna().sum())
display(
    df_final[
        ["Player Name","Player Team Name","Opponent Name","season","Gameweek","Opponent Difficulty"]
    ].head(20)
)


Unique seasons: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Max GW per season: {'2020-21': 38, '2021-22': 38, '2022-23': 38, '2023-24': 38, '2024-25': 38, '2025-26': 32}

Opponent Difficulty non-null: 171744


,Player Name,Player Team Name,Opponent Name,season,Gameweek,Opponent Difficulty
136097,Aaron Anselmino,Chelsea,Brighton,2024-25,25,3.0
136908,Aaron Anselmino,Chelsea,Aston Villa,2024-25,26,4.0
137691,Aaron Anselmino,Chelsea,Southampton,2024-25,27,1.0
138479,Aaron Anselmino,Chelsea,Leicester,2024-25,28,1.0
139137,Aaron Anselmino,Chelsea,Arsenal,2024-25,29,5.0
139909,Aaron Anselmino,Chelsea,Spurs,2024-25,30,2.0
140701,Aaron Anselmino,Chelsea,Brentford,2024-25,31,3.0
141694,Aaron Anselmino,Chelsea,Ipswich,2024-25,32,2.0
142918,Aaron Anselmino,Chelsea,Fulham,2024-25,33,3.0
143644,Aaron Anselmino,Chelsea,Everton,2024-25,34,3.0


In [123]:
print("SAFETY CHECK — Comparing column structure before merge")

OLD_TRAIN = "output/training_data_v2.csv"
NEW_TRAIN = "output/training_data_2025_26.csv"

df_old = pd.read_csv(OLD_TRAIN, encoding="utf-8-sig")
df_new = pd.read_csv(NEW_TRAIN, encoding="utf-8-sig")

print(f"   → Old dataset rows: {len(df_old):,}")
print(f"   → New dataset rows: {len(df_new):,}")

cols_old = set(df_old.columns)
cols_new = set(df_new.columns)

missing_in_new = cols_old - cols_new
missing_in_old = cols_new - cols_old

print("\n Columns missing in NEW dataset:", missing_in_new)
print(" Columns missing in OLD dataset:", missing_in_old)



SAFETY CHECK — Comparing column structure before merge
   → Old dataset rows: 171,905
   → New dataset rows: 24,974

 Columns missing in NEW dataset: {'Yellow Card', 'Red Cards', 'Player Name', 'Avg_Goals Conceded_L5', 'Opponent Name', 'Avg_Influence_L3', 'Avg_Creativity_L5', 'Avg_Influence_L5', 'Code', 'Player Team Name', 'Web Name', 'Avg_ICT Index_L5', 'Threat', 'Avg_ICT Index_L3', 'Creativity', 'Avg_Creativity_L3', 'Influence', 'Avg_Threat_L3', 'Avg_Goals Conceded_L3', 'Avg_Threat_L5'}
 Columns missing in OLD dataset: {'penalties_missed', 'recoveries', 'expected_goals_conceded', 'expected_assists', 'element_type', 'name', 'Player Team ID', 'in_dreamteam', 'clearances_blocks_interceptions', 'saves', 'Player Name Norm', 'bonus', 'threat', 'starts', 'own_goals', 'element', 'expected_goals', 'red_cards', 'tackles', 'Team_Points_Contribution_GW_Pct', 'was_home', 'influence', 'penalties_saved', 'expected_goal_involvements', 'yellow_cards', 'team', 'creativity', 'defensive_contribution', '

In [124]:

# SAFE MERGE — KEEP EXACT COLUMN ORDER FROM training_data_v2.csv

print(" Loading old + new datasets with SAFE column order rules...")

OLD_TRAIN = "output/training_data_v2.csv"
NEW_TRAIN = "output/training_data_2025_26.csv"
MERGED_OUT = "output/training_data_v3.csv"

df_old = pd.read_csv(OLD_TRAIN, encoding="utf-8-sig")
df_new = pd.read_csv(NEW_TRAIN, encoding="utf-8-sig")

print(f"   → Old dataset rows: {len(df_old):,}")
print(f"   → New dataset rows: {len(df_new):,}")


#Keep only the columns of the OLD file
# (the new dataset will be forced to follow this order)

old_cols = list(df_old.columns)

# Keep only the columns that exist in both
common_cols = [c for c in old_cols if c in df_new.columns]

print(f"Keeping {len(common_cols)} common columns in correct order")

df_old_clean = df_old[common_cols]
df_new_clean = df_new[common_cols]

# -----------------------------------------
# MERGE with correct column order
# -----------------------------------------
df_merged = pd.concat([df_old_clean, df_new_clean], ignore_index=True)

print(f" MERGED successfully → {len(df_merged):,} rows total")

# -----------------------------------------
# SAVE FINAL
# -----------------------------------------
df_merged.to_csv(MERGED_OUT, index=False, encoding="utf-8-sig")
print(f" Saved → {MERGED_OUT}")

# -----------------------------------------
#  VERIFY
# -----------------------------------------
print("\n Sanity check:")
print("Columns:", list(df_merged.columns))


 Loading old + new datasets with SAFE column order rules...
   → Old dataset rows: 171,905
   → New dataset rows: 24,974
Keeping 22 common columns in correct order
 MERGED successfully → 196,879 rows total
 Saved → output/training_data_v3.csv

 Sanity check:
Columns: ['Player UUID', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Total Points', 'ICT Index', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5']


In [125]:
print(" Checking duplicates (season + GW + Player UUID)...")

dups = df_merged[df_merged.duplicated(
    subset=["Player UUID", "season", "Gameweek"],
    keep=False
)]

print(f"Found {len(dups)} duplicated rows")
display(dups.head(20))


 Checking duplicates (season + GW + Player UUID)...
Found 75026 duplicated rows


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Position,Injury/Unavailable,Avg_Total Points_L3,Avg_Minutes Played_L3,Avg_Goals Scored_L3,Avg_Assists_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5
14,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,1,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,2,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,3,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,4,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,5,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,6,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,7,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
21,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,8,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,9,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,16b72858-75e4-4125-ad3b-e13f81e6d815,2025-26,10,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [126]:
df_merged = df_merged.drop_duplicates(
    subset=["Player UUID", "season", "Gameweek"],
    keep="first"
).reset_index(drop=True)

print(" Duplicates removed. New shape:", df_merged.shape)


 Duplicates removed. New shape: (151238, 22)


In [127]:
dups.groupby(["Player UUID","season","Gameweek"]).size().sort_values(ascending=False).head(20)




Player UUID                           season   Gameweek
3e20649c-3f65-4d75-998a-a137be556e95  2021-22  36          8
                                      2020-21  26          8
                                      2021-22  29          7
11b16d6f-11c5-4246-9766-b2b6d3f1c19c  2020-21  35          6
c445dcf2-e453-4549-87a4-c9977fa103fb  2025-26  26          6
9d4bab60-9257-4630-ae3b-4aabe590c687  2020-21  35          6
ced83cb7-c2b3-4453-9d62-0bf3f2d19edb  2020-21  35          6
de21d9ed-2379-46ef-8a6d-583bce9c3ac0  2020-21  35          6
a6f5936b-4b50-4aca-b910-9dab18059af5  2025-26  26          6
cf4a10bd-c897-4123-baa1-be57e82b399d  2020-21  35          6
051036aa-5efe-4691-995a-f816dc95b9b6  2025-26  26          6
a0136567-ef44-486e-899e-6e03ff50f83e  2020-21  35          6
d69c83b4-9ace-4ae7-ae73-1cb70da28e14  2020-21  35          6
c3337ae2-3ca0-48ad-b812-2679b954fd49  2025-26  26          6
ab237705-1b12-4bc9-b757-bd6c8b8e4726  2020-21  35          6
b74a658b-0105-41d0-8b8f-135d8

In [128]:
df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

dups = df[df.duplicated(["Player UUID", "season", "Gameweek"], keep=False)]
dups.sort_values(["Player UUID","season","Gameweek"]).head(200)


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Position,Injury/Unavailable,Avg_Total Points_L3,Avg_Minutes Played_L3,Avg_Goals Scored_L3,Avg_Assists_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5
52782,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,2020-21,19,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52783,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,2020-21,19,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52784,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,2020-21,19,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52791,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,2020-21,26,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52792,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,2020-21,26,0,0,0,0,0,0,0.0,...,DEF,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98348,00e8f0fc-d5d0-4d9d-b9e9-507b357185d2,2024-25,25,0,0,0,0,0,0,0.0,...,MID,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98349,00e8f0fc-d5d0-4d9d-b9e9-507b357185d2,2024-25,25,0,0,0,0,0,0,0.0,...,MID,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98356,00e8f0fc-d5d0-4d9d-b9e9-507b357185d2,2024-25,33,0,0,0,0,0,0,0.0,...,MID,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98357,00e8f0fc-d5d0-4d9d-b9e9-507b357185d2,2024-25,33,0,0,0,0,0,0,0.0,...,MID,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [129]:

import pandas as pd
from pathlib import Path

# 1. Define paths
TRAIN_FILE = Path("output/training_data_v3.csv")
MAPPING_FILE = Path("output/player_uuid_mapping.csv")

# 2. Load the training data and the mapping
df = pd.read_csv(TRAIN_FILE, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_FILE, encoding="utf-8-sig")

# 3. Find the UUID for Zidane Iqbal from the mapping file
# We search for "Zidane" in the 'Player Name Norm' or 'Player Name' columns of the mapping
iqbal_mapping = mapping[mapping["Player Name Norm"].str.contains("zidane", case=False, na=False)]

if not iqbal_mapping.empty:
    iqbal_uuid = iqbal_mapping["Player UUID"].iloc[0]
    print(f"Found UUID for Zidane Iqbal: {iqbal_uuid}")
    
    # 4. Filter the training data using this UUID and the season
    sub = df[
        (df["Player UUID"] == iqbal_uuid) & 
        (df["season"] == "2022-23")
    ]
    
    # 5. Save and Print
    if not sub.empty:
        sub.to_csv("output/iqbal_2022_23.csv", index=False, encoding="utf-8-sig")
        print(f"Found {len(sub)} rows for Zidane Iqbal in 2022-23.")
        print(sub)
    else:
        print("No match found for that UUID in the 2022-23 season data.")
else:
    print("Could not find Zidane Iqbal in the player_uuid_mapping.csv file.")

Found UUID for Zidane Iqbal: c10c39c3-d1a2-4c97-9bab-869dbdc207c2
Found 46 rows for Zidane Iqbal in 2022-23.
                                 Player UUID   season  Gameweek  \
171859  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23         1   
171860  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23         2   
171861  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23         3   
171862  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23         4   
171863  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23         5   
171864  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23         6   
171865  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23         9   
171866  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23        10   
171867  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23        11   
171868  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23        12   
171869  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23        13   
171870  c10c39c3-d1a2-4c97-9bab-869dbdc207c2  2022-23        14   
171871  c10c39c3-d1a

In [130]:
print("🧹 Fixing duplicate rows (Player UUID + season + GW)...")

df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

# Step 1 – Προτεραιότητα στα πραγματικά ματς (Minutes > 0)
df["PlayedFlag"] = df["Minutes Played"] > 0

# Step 2 – Βαθμολογούμε rows ώστε να ξέρουμε ποιο να κρατήσουμε
df["keep_rank"] = df.groupby(
    ["Player UUID", "season", "Gameweek"]
)["PlayedFlag"].transform(lambda s: s.rank(method="first", ascending=False))

# Step 3 – Κρατάμε ΜΟΝΟ το 1 καλύτερο
df_dedup = df[df["keep_rank"] == 1].drop(columns=["PlayedFlag", "keep_rank"])

print("✔️ Before:", len(df))
print("✔️ After :", len(df_dedup))
print("✔️ Removed duplicates:", len(df) - len(df_dedup))

df_dedup.to_csv("output/training_data_v3_clean.csv", index=False, encoding="utf-8-sig")
print("💾 Saved cleaned → training_data_v3_clean.csv")


🧹 Fixing duplicate rows (Player UUID + season + GW)...
✔️ Before: 196879
✔️ After : 151018
✔️ Removed duplicates: 45861
💾 Saved cleaned → training_data_v3_clean.csv


In [131]:
# Cell: FINAL SAFE VALIDATION (CLEAN VERSION)

import pandas as pd
import os

print("Loading cleaned training dataset...")
df = pd.read_csv("output/training_data_v3_clean.csv", encoding="utf-8-sig")
print(f"Rows: {len(df):,}")
print(f"Available Columns: {df.columns.tolist()}")

# 1. DUPLICATES CHECK
print("Checking duplicates (Player UUID + season + GW)...")
if "Player UUID" in df.columns:
    dupes = df.duplicated(subset=["Player UUID","season","Gameweek"], keep=False)
    dupe_rows = df[dupes]
    print(f"Duplicate rows found: {len(dupe_rows):,}")
    if len(dupe_rows) > 0:
        display(dupe_rows.head(20))
    else:
        print("No duplicates detected.")
else:
    print("Skip: 'Player UUID' not found.")

# 2. MISSING OPPONENT DIFFICULTY
print("Checking missing Opponent Difficulty...")
if "Opponent Difficulty" in df.columns:
    missing_od = df[df["Opponent Difficulty"].isna()]
    print(f"Missing OD count: {len(missing_od):,}")
    if len(missing_od) > 0:
        display(missing_od.head(20))
    else:
        print("No missing OD values.")
else:
    print("Skip: 'Opponent Difficulty' not found.")

# 3. GAMEWEEK GAPS
print("Checking gameweek gaps per player...")
if "Player UUID" in df.columns:
    gap_examples = []
    # Check a sample of 100 players
    sample_pids = df["Player UUID"].unique()[:100] 
    for pid in sample_pids:
        for season in df[df["Player UUID"] == pid]["season"].unique():
            sub = df[(df["Player UUID"] == pid) & (df["season"] == season)]
            gws = sorted(sub["Gameweek"].unique())
            expected = list(range(min(gws), max(gws)+1))
            if gws != expected:
                gap_examples.append((pid, season, gws))
                
    print(f"Players with gaps in sample: {len(gap_examples)}")
    print("Note: Gaps are expected as players may miss matches.")
else:
    print("Skip: 'Player UUID' not found.")

# 4. TEAM NAME CONSISTENCY CHECK
print("Checking team name consistency...")
if "Player Team Name" in df.columns:
    team_file = "data/2022-23/teams.csv"
    if os.path.exists(team_file):
        teams_df = pd.read_csv(team_file, encoding="utf-8-sig")
        name_col = "name" if "name" in teams_df.columns else teams_df.columns[0]
        unique_team_names = sorted(teams_df[name_col].unique())

        invalid_teams = df[~df["Player Team Name"].isin(unique_team_names)]
        print(f"Invalid team names: {len(invalid_teams):,}")
        if len(invalid_teams) > 0:
            display(invalid_teams.head(20))
        else:
            print("All team names match known Premier League teams.")
    else:
        print(f"Skip: Reference file {team_file} not found.")
else:
    print("Info: 'Player Team Name' not in this CSV. Skipping team check.")

# 5. OPPONENT NAME CONSISTENCY
print("Checking opponent name consistency...")
if "Opponent Name" in df.columns:
    invalid_opps = df[~df["Opponent Name"].isin(unique_team_names)]
    print(f"Invalid opponent names: {len(invalid_opps):,}")
    if len(invalid_opps) > 0:
        display(invalid_opps.head(20))
    else:
        print("All opponent names valid.")
else:
    print("Info: 'Opponent Name' not in this CSV. Skipping opponent check.")

print("VALIDATION COMPLETE")

Loading cleaned training dataset...
Rows: 151,018
Available Columns: ['Player UUID', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Total Points', 'ICT Index', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5']
Checking duplicates (Player UUID + season + GW)...
Duplicate rows found: 0
No duplicates detected.
Checking missing Opponent Difficulty...
Missing OD count: 161


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Position,Injury/Unavailable,Avg_Total Points_L3,Avg_Minutes Played_L3,Avg_Goals Scored_L3,Avg_Assists_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5
1301,6da20176-efb4-4639-9809-8efd4495c95e,2025-26,31,0,0,0,0,0,0,0.0,...,DEF,0.0,1.000000,34.333333,0.000000,0.000000,1.0,35.6,0.0,0.0
1502,3ecf98f9-89d3-4172-934b-4599a5742505,2025-26,31,0,0,0,0,0,0,0.0,...,FWD,0.0,3.333333,78.333333,0.333333,0.333333,3.2,95.4,0.2,0.2
2335,8ff95a0c-fee4-480b-97c8-1f10e4948775,2025-26,31,0,0,0,0,0,0,0.0,...,MID,0.0,4.333333,59.000000,0.000000,0.666667,4.4,69.8,0.0,0.6
3373,26519591-834b-46dd-b94a-342a5f4f69aa,2025-26,31,0,0,0,0,0,0,0.0,...,MID,1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
5776,a0f6c8f2-3970-434f-9d44-643ddb931bf8,2025-26,31,0,0,0,0,0,0,0.0,...,DEF,1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
7912,472c89bd-a8b4-4404-85e8-a4a4fdf7e673,2025-26,31,0,0,0,0,0,0,0.0,...,MID,0.0,6.333333,89.333333,0.333333,0.000000,4.8,89.6,0.2,0.0
9334,bad7dfdb-a980-47b0-b2d4-33a3f097d88b,2025-26,31,0,0,0,0,0,0,0.0,...,MID,0.0,0.666667,35.000000,0.000000,0.000000,1.4,40.8,0.0,0.0
10548,e2d76277-7e9c-4ec9-8075-02474349e475,2025-26,31,0,0,0,0,0,0,0.0,...,MID,0.0,5.666667,84.666667,0.666667,0.000000,7.0,86.0,0.6,0.2
12018,dab4974b-7610-401a-bc42-86bc70c9fbb3,2025-26,31,0,0,0,0,0,0,0.0,...,MID,1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
15926,450bb2fa-45ce-4d8d-b428-6481c1227a97,2025-26,31,0,0,0,0,0,0,0.0,...,DEF,1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0


Checking gameweek gaps per player...
Players with gaps in sample: 151
Note: Gaps are expected as players may miss matches.
Checking team name consistency...
Info: 'Player Team Name' not in this CSV. Skipping team check.
Checking opponent name consistency...
Info: 'Opponent Name' not in this CSV. Skipping opponent check.
VALIDATION COMPLETE


In [132]:
import numpy as np
import pandas as pd
from pathlib import Path
import re
from unidecode import unidecode

def normalize_player_name(name):
    if pd.isna(name): return ""
    name = str(name).strip().lower()
    name = unidecode(name)
    name = re.sub(r"[\s_]*\d+\s*$", "", name)
    name = name.replace("_", " ")
    name = "".join(c for c in name if c.isalnum() or c.isspace())
    return " ".join(name.split())

print("Loading datasets...")
df = pd.read_csv("output/training_data_v3_clean.csv", encoding="utf-8-sig")
mapping = pd.read_csv("output/player_uuid_mapping.csv")

# 1. Build a Season-Agnostic Bridge for 2025-26
print("Building specialized 2025-26 Bridge...")
pr_2025_path = Path("data/2025-26/players_raw.csv")
if not pr_2025_path.exists():
    raise FileNotFoundError("Could not find data/2025-26/players_raw.csv")

pr_2025 = pd.read_csv(pr_2025_path)
id_col = "element" if "element" in pr_2025.columns else "id"

# Create normalization for 2025-26 specifically
pr_2025["norm_full"] = (pr_2025["first_name"].fillna("") + " " + pr_2025["second_name"].fillna("")).apply(normalize_player_name)
pr_2025["norm_web"] = pr_2025["web_name"].fillna("").apply(normalize_player_name)

# Create a lookup dictionary for 2025-26: Name -> (Code, Team)
lookup_2025 = {}
for _, r in pr_2025.iterrows():
    lookup_2025[r["norm_full"]] = (r[id_col], r["team"])
    lookup_2025[r["norm_web"]] = (r[id_col], r["team"])

# 2. Build Historical Bridge (2020-2025)
print("Building Historical Bridge...")
bridge_list = []
for season_folder in Path("data").glob("20*"):
    if season_folder.name == "2025-26": continue
    pr_path = season_folder / "players_raw.csv"
    if pr_path.exists():
        pr = pd.read_csv(pr_path)
        cid = "id" if "id" in pr.columns else "element"
        pr["norm_full"] = (pr["first_name"].fillna("") + " " + pr["second_name"].fillna("")).apply(normalize_player_name)
        temp = pr[["norm_full", cid, "team"]].copy()
        temp.columns = ["Player Name Norm", "Code", "Player Team ID"]
        temp["season"] = season_folder.name
        bridge_list.append(temp)

hist_bridge = pd.concat(bridge_list).drop_duplicates(["Player Name Norm", "season"])

# 3. Apply Links to Training Data
print("Applying Links...")
if "Player Name Norm" not in mapping.columns:
    mapping["Player Name Norm"] = mapping["Player Name"].apply(normalize_player_name)
if "Web Name" in mapping.columns:
    mapping["Web Name Norm"] = mapping["Web Name"].apply(normalize_player_name)
else:
    mapping["Web Name Norm"] = mapping["Player Name Norm"]

# Map UUID to Names
df = df.merge(mapping[["Player UUID", "Player Name Norm", "Web Name Norm"]], on="Player UUID", how="left")

# Link 2025-26 rows using the specialized lookup
mask_2025 = df["season"] == "2025-26"
def get_2025_data(row):
    # Try full name then web name
    res = lookup_2025.get(row["Player Name Norm"])
    if not res:
        res = lookup_2025.get(row["Web Name Norm"])
    return pd.Series(res) if res else pd.Series([np.nan, np.nan])

print("Mapping 2025-26 data specifically...")
df.loc[mask_2025, ["Code", "Player Team ID"]] = df[mask_2025].apply(get_2025_data, axis=1).values

# Link Historical rows
print("Mapping Historical data...")
df = df.merge(hist_bridge, on=["Player Name Norm", "season"], how="left", suffixes=("", "_hist"))
df["Code"] = df["Code"].fillna(df["Code_hist"])
df["Player Team ID"] = df["Player Team ID"].fillna(df["Player Team ID_hist"])
df.drop(columns=["Code_hist", "Player Team ID_hist"], inplace=True)

# Final Check
missing_count = df["Player Team ID"].isna().sum()
if missing_count > 0:
    print(f"Missing {missing_count} rows.")
    print(df[df["Player Team ID"].isna()]["season"].value_counts())
    df["Player Team ID"] = df["Player Team ID"].fillna(0).astype(int)
else:
    print("Success: 100% of rows matched!")

# 4. Team Contributions
print("Calculating Team Contributions...")
team_stats = df.groupby(["Player Team ID", "season", "Gameweek"])["Total Points"].sum().reset_index()
team_stats.rename(columns={"Total Points": "Team_Total_Points_GW"}, inplace=True)
team_stats["Team_Total_Points_CUM"] = team_stats.groupby(["Player Team ID", "season"])["Team_Total_Points_GW"].cumsum()
team_stats["Team_Total_Points"] = team_stats.groupby(["Player Team ID", "season"])["Team_Total_Points_CUM"].shift(1).fillna(0)

df = df.merge(team_stats, on=["Player Team ID", "season", "Gameweek"], how="left")
df["Player_Season_Points"] = df.groupby(["Player UUID", "season"])["Total Points"].cumsum().shift(1).fillna(0)

df["Team_Points_Contribution_GW_Pct"] = np.where(df["Team_Total_Points_GW"] > 0, (df["Total Points"] / df["Team_Total_Points_GW"] * 100), 0).round(2)
df["Team_Points_Contribution_Causal_Pct"] = np.where(df["Team_Total_Points"] > 0, (df["Player_Season_Points"] / df["Team_Total_Points"] * 100), 0).round(2)

df.to_csv("output/training_data_v4_with_contrib.csv", index=False, encoding="utf-8-sig")
print("DONE!")

Loading datasets...
Building specialized 2025-26 Bridge...
Building Historical Bridge...
Applying Links...
Mapping 2025-26 data specifically...
Mapping Historical data...
Missing 2334 rows.
season
2024-25    447
2021-22    443
2023-24    440
2022-23    391
2020-21    325
2025-26    288
Name: count, dtype: int64
Calculating Team Contributions...
DONE!


In [133]:
import pandas as pd
from pathlib import Path

# Check the current season's raw file
pr_2025 = pd.read_csv("data/2025-26/players_raw.csv")
print("Columns in 2025-26 players_raw:", pr_2025.columns.tolist())
print("\nFirst 3 rows of 2025-26 data:")
print(pr_2025.head(3))

Columns in 2025-26 players_raw: ['element', 'team', 'element_type', 'first_name', 'second_name', 'web_name']

First 3 rows of 2025-26 data:
   element  team  element_type first_name            second_name      web_name
0        1     1             1      David            Raya Martín          Raya
1        2     1             1       Kepa  Arrizabalaga Revuelta  Arrizabalaga
2        3     1             1       Karl                   Hein          Hein


In [134]:
import numpy as np
import pandas as pd

# Load the file with low_memory=False to handle potential mixed types in index columns
df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig", low_memory=False)

print("Loaded:", len(df), "rows")

# Columns produced by the v4 Golden Cell
required_cols = [
    "Team_Total_Points_GW", "Team_Total_Points", "Player_Season_Points",
    "Team_Points_Contribution_GW_Pct", "Team_Points_Contribution_Causal_Pct",
    "Team_Contribution_Rank_GW"
]

print("Checking required columns...")
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print("Missing columns:", missing)
    print("Available columns are:", df.columns.tolist())
else:
    print("All contribution columns present.")

# Check for NaNs
print("Checking NaNs in contribution cols...")
existing_req = [c for c in required_cols if c in df.columns]
nan_counts = df[existing_req].isna().sum()
if nan_counts.sum() > 0:
    print(nan_counts[nan_counts > 0])
else:
    print("No NaNs detected.")

# Check GW1 causal logic
print("Checking Gameweek 1 causal values (Expect 0)...")
gw1 = df[df["Gameweek"] == 1]
print("Team_Total_Points > 0 on GW1:", (gw1["Team_Total_Points"] > 0).sum())
print("Player_Season_Points > 0 on GW1:", (gw1["Player_Season_Points"] > 0).sum())

# Check sum of GW contributions
team_col = "Player Team ID" if "Player Team ID" in df.columns else "team"
if team_col in df.columns:
    print(f"Checking if GW contributions sum to ~100% per team/week (using {team_col})...")
    group_sum = df.groupby([team_col, "season", "Gameweek"])["Team_Points_Contribution_GW_Pct"].sum()
    group_sum = group_sum[group_sum > 0]
    bad_groups = group_sum[(group_sum < 99.9) | (group_sum > 100.1)]
    print("Groups failing 100% sum check:", len(bad_groups))
    if len(bad_groups) > 0:
        print(bad_groups.head())
else:
    print("Skip: Team identifier column not found.")

# Summary statistics
print("Contribution percentage summary:")
display(df[["Team_Points_Contribution_GW_Pct", "Team_Points_Contribution_Causal_Pct"]].describe())

print("Top 10 Season-Long Contributors (Causal Pct):")
display(df.sort_values("Team_Points_Contribution_Causal_Pct", ascending=False).head(10))

Loaded: 153352 rows
Checking required columns...
Missing columns: ['Team_Contribution_Rank_GW']
Available columns are: ['Player UUID', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Total Points', 'ICT Index', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5', 'Player Name Norm', 'Web Name Norm', 'Code', 'Player Team ID', 'Team_Total_Points_GW', 'Team_Total_Points_CUM', 'Team_Total_Points', 'Player_Season_Points', 'Team_Points_Contribution_GW_Pct', 'Team_Points_Contribution_Causal_Pct']
Checking NaNs in contribution cols...
No NaNs detected.
Checking Gameweek 1 causal values (Expect 0)...
Team_Total_Points > 0 on GW1: 0
Player_Season_Points > 0 on GW1: 2665
Checking if GW contributions sum to ~100% per team/week (using Player Team ID)...
Gro

,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct
count,153352.000000,153352.000000
mean,2.927242,3.466113
std,5.750912,39.497358
min,-150.000000,-11.760000
25%,0.000000,0.000000
50%,0.000000,1.270000
75%,4.260000,5.070000
max,150.000000,9000.000000


Top 10 Season-Long Contributors (Causal Pct):


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Player Name Norm,Web Name Norm,Code,Player Team ID,Team_Total_Points_GW,Team_Total_Points_CUM,Team_Total_Points,Player_Season_Points,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct
79725,a3e5306c-3429-408d-b49b-6d25c36a3512,2020-21,2,28,0,0,0,0,1,2.3,...,keinan davis,keinan davis,50.0,2,66,67,1.0,90.0,1.52,9000.0
141980,867215ca-abbb-4659-a6e0-83c23500be09,2020-21,2,0,0,0,0,0,0,0.0,...,wesley moraes,wesley moraes,48.0,2,66,67,1.0,58.0,0.00,5800.0
36439,529502b2-ce84-4366-9ad5-76b474a1ffaf,2020-21,2,90,0,0,1,0,3,5.9,...,douglas luiz soares de paulo,douglas luiz soares de paulo,52.0,2,66,67,1.0,55.0,4.55,5500.0
74886,1b2e156b-56dc-478d-835d-f81fd1e5eede,2020-21,2,0,0,0,0,0,0,0.0,...,jose ignacio peleteiro romallo,jose ignacio peleteiro romallo,34.0,2,66,67,1.0,44.0,0.00,4400.0
66870,92db56ab-7c1f-4e4f-9c0e-1dbc534ee1a5,2020-21,2,0,0,0,0,0,0,0.0,...,jed steer,jed steer,32.0,2,66,67,1.0,40.0,0.00,4000.0
108436,e9e859cd-d8af-4c9f-8537-e2964550359d,2020-21,2,0,0,0,0,0,0,0.0,...,neil taylor,neil taylor,31.0,2,66,67,1.0,40.0,0.00,4000.0
17141,6391a5bf-2199-4a3f-9617-0560c9af1707,2020-21,2,0,0,0,0,0,0,0.0,...,bjorn engels,bjorn engels,36.0,2,66,67,1.0,39.0,0.00,3900.0
56232,11b5f5a7-4b3f-4aed-8f48-81d089cabc8a,2020-21,2,0,0,0,0,0,0,0.0,...,indiana vassilev,indiana vassilev,51.0,2,66,67,1.0,28.0,0.00,2800.0
27476,f91b2c57-590c-4f04-9663-1aed45a03a3f,2020-21,2,61,0,0,1,0,3,4.7,...,conor hourihane,conor hourihane,33.0,2,66,67,1.0,22.0,4.55,2200.0
112069,c98696d5-6ee3-48cb-8491-7bf3c66aba03,2020-21,2,90,0,0,1,0,2,5.6,...,ollie watkins,ollie watkins,514.0,2,66,67,1.0,21.0,3.03,2100.0


In [135]:
# Load the current dataset
file_path = "output/training_data_v4_with_contrib.csv"
df = pd.read_csv(file_path, low_memory=False)

print(f"Starting cleanup on {len(df)} rows...")

# 1. Force Causal/Historical features to 0 for Gameweek 1
# This ensures a clean start for every season
causal_cols = ["Team_Total_Points", "Player_Season_Points", "Team_Points_Contribution_Causal_Pct"]

for col in causal_cols:
    if col in df.columns:
        df.loc[df["Gameweek"] == 1, col] = 0

# 2. Re-calculate Team_Contribution_Rank_GW
# We rank players within their team for each gameweek (1.0 = top contributor)
if "Team_Points_Contribution_GW_Pct" in df.columns:
    print("Calculating Team_Contribution_Rank_GW...")
    df["Team_Contribution_Rank_GW"] = df.groupby(["Player Team ID", "season", "Gameweek"])["Team_Points_Contribution_GW_Pct"].rank(
        method="min", 
        ascending=False, 
        pct=True
    ).round(4)

# 3. Clean up Team ID 0 artifacts
# If a player is unmatched (Team ID 0), we set their contribution metrics to 0
mask_unmatched = df["Player Team ID"] == 0
df.loc[mask_unmatched, ["Team_Points_Contribution_GW_Pct", "Team_Contribution_Rank_GW"]] = 0

# 4. Final Formatting
# Ensure all contribution columns are rounded for cleanliness
cols_to_round = ["Team_Points_Contribution_GW_Pct", "Team_Points_Contribution_Causal_Pct"]
for col in cols_to_round:
    if col in df.columns:
        df[col] = df[col].round(2)

# Save the polished version
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print("Cleanup complete!")
print("Causal GW1 values reset to 0.")
print("Team_Contribution_Rank_GW has been added.")
print(f"File saved to: {file_path}")

Starting cleanup on 153352 rows...
Calculating Team_Contribution_Rank_GW...
Cleanup complete!
Causal GW1 values reset to 0.
Team_Contribution_Rank_GW has been added.
File saved to: output/training_data_v4_with_contrib.csv


In [136]:
# Load the dataset
file_path = "output/training_data_v4_with_contrib.csv"
df = pd.read_csv(file_path, low_memory=False)

print(f"Initial row count: {len(df):,}")

# 1. Remove rows where Position is missing (mostly Managers)
df_clean = df.dropna(subset=['Position']).copy()

# 2. Convert Position to categorical to save memory and prepare for ML
df_clean['Position'] = df_clean['Position'].astype('category')

print(f"Rows removed: {len(df) - len(df_clean)}")
print(f"Final clean row count: {len(df_clean):,}")

# 3. Final check: Are there any other NaNs?
nan_summary = df_clean.isna().sum()
cols_with_nans = nan_summary[nan_summary > 0]

if not cols_with_nans.empty:
    print("Remaining columns with NaNs:")
    print(cols_with_nans)
else:
    print("Success: No missing values remain in the dataset.")

# Save the finalized training set
df_clean.to_csv("output/final_training_data_v5.csv", index=False, encoding="utf-8-sig")
print("Saved to: output/final_training_data_v5.csv")

Initial row count: 153,352
Rows removed: 312
Final clean row count: 153,040
Remaining columns with NaNs:
Opponent Difficulty     165
Is Home                 165
Code                   2334
dtype: int64
Saved to: output/final_training_data_v5.csv


In [137]:
df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig", low_memory=False)

print("Loaded:", len(df), "rows")

# 1. Missing Position count
missing_pos = df["Position"].isna().sum()
print(f"Missing Position values: {missing_pos}")

# 2. Sample rows with missing Position
print("Sample rows with missing Position:")
display(df[df["Position"].isna()].head(20))

# 3. Frequency per season
# Using Player UUID as the count column since Player Name is missing
print("Missing positions per season:")
if "season" in df.columns:
    display(df[df["Position"].isna()]
            .groupby("season")["Player UUID"]
            .count()
            .sort_index())

# 4. Frequency per Team ID
# Using Player Team ID as the grouping column
print("Missing positions per Player Team ID:")
if "Player Team ID" in df.columns:
    display(df[df["Position"].isna()]
            .groupby("Player Team ID")["Player UUID"]
            .count()
            .sort_values(ascending=False)
            .head(20))

# 5. element_type breakdown
if "element_type" in df.columns:
    print("element_type breakdown for missing Position:")
    display(
        df[df["Position"].isna()]
        .groupby("element_type")["Player UUID"]
        .count()
    )
else:
    print("No element_type column found in the dataset.")

Loaded: 153352 rows
Missing Position values: 312
Sample rows with missing Position:


,Player UUID,season,Gameweek,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Total Points,ICT Index,...,Web Name Norm,Code,Player Team ID,Team_Total_Points_GW,Team_Total_Points_CUM,Team_Total_Points,Player_Season_Points,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct,Team_Contribution_Rank_GW
7492,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,23,0,0,0,0,0,13,0.0,...,andoni iraola,738.0,3,106,1089,983.0,0.0,12.26,0.00,0.0488
7493,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,24,0,0,0,0,0,0,0.0,...,andoni iraola,738.0,3,19,1108,1089.0,13.0,0.00,1.19,0.3171
7494,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,25,0,0,0,0,0,9,0.0,...,andoni iraola,738.0,3,61,1169,1108.0,13.0,14.75,1.17,0.0488
7495,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,26,0,0,0,0,0,0,0.0,...,andoni iraola,738.0,3,18,1187,1169.0,22.0,0.00,1.88,0.3415
7496,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,27,0,0,0,0,0,1,0.0,...,andoni iraola,738.0,3,32,1219,1187.0,22.0,3.12,1.85,0.1707
7497,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,28,0,0,0,0,0,5,0.0,...,andoni iraola,738.0,3,42,1261,1219.0,23.0,11.90,1.89,0.0732
7498,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,29,0,0,0,0,0,1,0.0,...,andoni iraola,738.0,3,24,1285,1261.0,28.0,4.17,2.22,0.1707
7499,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,30,0,0,0,0,0,1,0.0,...,andoni iraola,738.0,3,29,1314,1285.0,29.0,3.45,2.26,0.1220
7500,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,31,0,0,0,0,0,5,0.0,...,andoni iraola,738.0,3,44,1358,1314.0,30.0,11.36,2.28,0.0732
7501,5d064b95-d989-43d1-a43b-80bc7f325c38,2024-25,32,0,0,0,0,0,9,0.0,...,andoni iraola,738.0,3,64,1422,1358.0,35.0,14.06,2.58,0.0732


Missing positions per season:


season
2024-25    312
Name: Player UUID, dtype: int64

Missing positions per Player Team ID:


Player Team ID
4     16
3     16
6     16
5     16
19    16
20    16
8     16
9     16
11    16
10    16
16    16
14    16
18    16
17    16
15    15
1     15
13    15
12    15
2     14
7     14
Name: Player UUID, dtype: int64

No element_type column found in the dataset.


In [138]:
import pandas as pd
from pathlib import Path

print("📂 Loading training_data_v4_with_contrib.csv ...")
df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig")
print("   → Rows:", len(df))

# ------------------------------------------------------------
# 1️⃣ Load players_raw for ALL seasons
# ------------------------------------------------------------

DATA_ROOT = Path("data")

players_raw_list = []

for season_folder in sorted(DATA_ROOT.iterdir()):
    pr_path = season_folder / "players_raw.csv"
    if pr_path.exists():
        print(f"📥 Loading → {pr_path}")
        temp = pd.read_csv(pr_path)

        # normalize ID column
        if "element" in temp.columns:
            temp = temp.rename(columns={"element": "Code"})
        elif "id" in temp.columns:
            temp = temp.rename(columns={"id": "Code"})
        else:
            continue

        if "element_type" not in temp.columns:
            print("⚠ Missing element_type → skipping")
            continue

        temp = temp[["Code", "element_type"]]
        temp["Code"] = pd.to_numeric(temp["Code"], errors="coerce")
        players_raw_list.append(temp)

players_raw = pd.concat(players_raw_list, ignore_index=True).drop_duplicates("Code")

print("✔ Total players_raw combined:", len(players_raw))

# ------------------------------------------------------------
# 2️⃣ Merge positions back into training dataset
# ------------------------------------------------------------

df["Code"] = pd.to_numeric(df["Code"], errors="coerce")

df_fix = df.merge(players_raw, on="Code", how="left")

POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
df_fix["Position"] = df_fix["element_type"].map(POS_MAP)

missing_after = df_fix["Position"].isna().sum()
print("\n🔍 Missing positions AFTER FIX:", missing_after)

# ------------------------------------------------------------
# 3️⃣ Save final fixed version
# ------------------------------------------------------------
out_path = "output/training_data_v6_positions_fixed.csv"
df_fix.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n💾 Saved → {out_path}")


📂 Loading training_data_v4_with_contrib.csv ...
   → Rows: 153352
📥 Loading → data\2018-19\players_raw.csv
📥 Loading → data\2019-20\players_raw.csv
📥 Loading → data\2020-21\players_raw.csv
📥 Loading → data\2021-22\players_raw.csv
📥 Loading → data\2022-23\players_raw.csv
📥 Loading → data\2023-24\players_raw.csv
📥 Loading → data\2024-25\players_raw.csv
📥 Loading → data\2025-26\players_raw.csv
✔ Total players_raw combined: 866

🔍 Missing positions AFTER FIX: 2334

💾 Saved → output/training_data_v6_positions_fixed.csv


In [139]:
# %% [Cell: Final Processing - Cleaning Ghost Players & 0-Min Rows]

# 1. Define final output path
CLEAN_OUT = OUTPUT_DIR / "dataset_no0min.csv"

print(f"Preparing final dataset from seasons: {df_final['season'].unique()}")

# --- STEP A: REMOVE GHOST PLAYERS (The 2025-26 Logic) ---
# We identify players who have 0 total minutes in the current season.
# If a player hasn't touched the pitch by GW29, they shouldn't be in our forecast.

current_season_mask = df_final["season"] == "2025-26"
active_this_season = df_final[current_season_mask].groupby("Player UUID")["Minutes Played"].sum()
active_uuids = active_this_season[active_this_season > 0].index

# Keep historical data, but for the current season, only keep players with > 0 total mins
df_filtered = df_final[
    (~current_season_mask) | (df_final["Player UUID"].isin(active_uuids))
].copy()

# --- STEP B: FILTER INACTIVE ROWS (Minutes > 0) ---
# Filter to keep only rows where players actually played in those specific weeks
df_dataset_no0min = df_filtered[df_filtered["Minutes Played"] > 0].copy()

print(f"Original records: {len(df_final):,}")
print(f"Records after Ghost Removal: {len(df_filtered):,}")
print(f"Final records (Minutes > 0): {len(df_dataset_no0min):,}")

# Save to output folder
df_dataset_no0min.to_csv(CLEAN_OUT, index=False, encoding="utf-8-sig")

print(f"SUCCESS!")
print(f"File saved to: {CLEAN_OUT}")

Preparing final dataset from seasons: ['2024-25' '2025-26' '2020-21' '2021-22' '2023-24' '2022-23']
Original records: 171,905
Records after Ghost Removal: 158,583
Final records (Minutes > 0): 67,463
SUCCESS!
File saved to: output\dataset_no0min.csv
